![DataKern Watermark](https://img.shields.io/badge/DATAKERN-Spark%20Practice%20Lab-F26522?style=for-the-badge)

# Lab 1 · Read: get the data in correctly

> ## 🎯 The idea of this lab
> This lab is **not** a test of whether you can invent Spark code. Its goal is to build **developer memory**: the way a working data engineer handles Spark.
> *Look at the data → write a transformation → run it → check the result → fix what is wrong → move on.*
> You repeat that loop until the core moves come from your **fingers**, not from a search box.
>
> Almost all the code is given. You **retype it and run it**, and after every run you **write down what you observed**. Typing builds the muscle. Observing turns the muscle into understanding.

**Time:** about 60 to 75 minutes · **Needs:** the Olist CSV files, Java 17+, `pip install -r requirements.txt`

### The scenario
You have just joined the Olist data team. Your lead sends you tickets, one at a time. This notebook has four. Each is about the first job of every pipeline: **reading the data correctly**. Most silent data loss starts here: a wrong type, a split row, a header counted as data. Nothing crashes and the numbers are simply wrong.

### How a ticket works
| Symbol | What you do |
|---|---|
| ⌨️ **Type this** | Retype the code shown into the empty cell and run it. Do not paste. |
| 📝 **Observe** | Look at the output and write what you saw in your own words, right under it. |
| ✍️ **Now you** | The same pattern on new input. Write it yourself, with the typed code above as your model. |
| ✅ **Check** | Run it. A green tick means your result matches the known answer. |

### Three habits to start building today
1. **Type it, don't paste it.** Your fingers learn what your eyes only skim.
2. **Look, do, check.** After every transformation run the same three lines: `df.printSchema()`, `df.show(5)`, `df.count()`.
3. **Write down what you saw.** A number you did not write down is a number you did not notice.

> **Run the cells top to bottom.** When you finish: *Restart* the kernel and *Run All*. No cell should show an error. Use the exact variable names given, because the checks look for them.

---
## Setup: start Spark and point to the files
Run these two cells (nothing to type).

In [ ]:
from pyspark.sql import functions as F
from lab_utils import get_spark, olist, check, checks_summary, root_cause, JobCounter

spark = get_spark("Lab 1 · Read")

In [ ]:
ORDERS    = olist("olist_orders_dataset.csv")
CUSTOMERS = olist("olist_customers_dataset.csv")
SELLERS   = olist("olist_sellers_dataset.csv")
REVIEWS   = olist("olist_order_reviews_dataset.csv")

> **First run only:** Spark downloads the Delta Lake jars (1 to 2 minutes, needs internet) and prints red text. That is normal.
> **Open the Spark UI link** printed above in a browser tab and keep it open for the whole lab.
> Write the Spark version and the number of cores at the top of your `OBSERVATION_REPORT.md`.

---
## 🎫 Ticket 1 · "Load the orders and tell me what you see"

> **Lead:** *"Load orders.csv. How many orders do we have, and what does one row look like?"*

**Issue:** a CSV is just text. Spark cannot know that line 1 holds column names or what type each column is.
**See:** what you get with no options, then with the one option that fixes the names.

⌨️ **Type this.** Type this into the cell below (do not paste), then run it with Shift+Enter:

```python
raw = spark.read.csv(ORDERS)
raw.printSchema()
raw.show(3, truncate=28)
raw.count()
```

In [ ]:
# ⌨️ type it here

📝 **Observe.** What are the columns called? Where did the header line go? What type is every column? What does `count()` say, and is that the number of orders?

> **I saw:** *(double-click this cell, replace this line with your own words, press Shift+Enter)*

⌨️ **Type this.** Type this into the cell below (do not paste), then run it with Shift+Enter:

```python
orders = spark.read.csv(ORDERS, header=True)
orders.printSchema()
orders.show(3, truncate=28)
orders.count()
```

In [ ]:
# ⌨️ type it here

📝 **Observe.** What changed compared with the first read: the column names, the row count, the types? Which problem is **still there**? (Look at the types, and think about `order_purchase_timestamp`.)

> **I saw:** *(double-click this cell, replace this line with your own words, press Shift+Enter)*

**Break it on purpose.** Every developer mistypes a column name. Reading the error well is a skill, so let us cause one. The `try/except` is only there so *Run All* keeps going; in real life you would simply see a red box.

⌨️ **Type this.** Type this into the cell below (do not paste), then run it with Shift+Enter:

```python
try:
    orders.select("order_idd")
except Exception as e:
    print(type(e).__name__)
    print(str(e)[:330])
```

In [ ]:
# ⌨️ type it here

📝 **Observe.** Which part of the message names the problem? What did Spark **suggest**? If it had not suggested anything, what is the quickest way to see the real column names?

> **I saw:** *(double-click this cell, replace this line with your own words, press Shift+Enter)*

✍️ **Now you.** 
Load `customers.csv` (the path is in `CUSTOMERS`) with its header into a variable called **`customers`**. Then do the three-line habit on it: `printSchema()`, `show(3)`, `count()`.

In [ ]:
# ✍️ your turn: write it here

📝 **Observe.** Did it work first time? If you hit an error, what was it and how did you fix it? (Add it to the bug list in your report.)

> **I saw:** *(double-click this cell, replace this line with your own words, press Shift+Enter)*

In [ ]:
check("orders: column names come from the header", lambda: orders.columns[0] == "order_id",
      "use the option that treats line 1 as column names")
check("orders: 99,441 rows (the header is not an order)", lambda: orders.count() == 99_441)
check("customers: loaded with header, 99,441 rows",
      lambda: customers.columns[0] == "customer_id" and customers.count() == 99_441,
      "variable must be called customers, read with header=True")

---
## 🎫 Ticket 2 · "The date columns are strings. Give me real types. Don't lose anything."

> **Lead:** *"Everything Spark gave us is a string. I need proper types. And last time someone's zip codes got mangled, so be careful."*

**Issue:** `header=True` fixed the names, but every column is still a string. Spark has two ways to get types: `inferSchema=True` (it reads your data to guess) or an explicit `schema` (you declare it).
**See:** what each way costs, and what the guess does to a zip code.

⌨️ **Type this.** Type this into the cell below (do not paste), then run it with Shift+Enter:

```python
CUSTOMER_DDL = ("customer_id STRING, customer_unique_id STRING, customer_zip_code_prefix STRING, "
                "customer_city STRING, customer_state STRING")

with JobCounter(spark) as j_infer:
    customers_inferred = spark.read.csv(CUSTOMERS, header=True, inferSchema=True)

with JobCounter(spark) as j_explicit:
    customers_explicit = spark.read.csv(CUSTOMERS, header=True, schema=CUSTOMER_DDL)

print("jobs launched by inferSchema:     ", j_infer.jobs)
print("jobs launched by an explicit schema:", j_explicit.jobs)
customers_inferred.printSchema()
```

In [ ]:
# ⌨️ type it here

📝 **Observe.** How many jobs did each read launch **on the read line itself**? What do you think Spark was doing in those jobs? What type did `inferSchema` pick for `customer_zip_code_prefix`?

> **I saw:** *(double-click this cell, replace this line with your own words, press Shift+Enter)*

⌨️ **Type this.** Type this (a chained query: each line adds one step; the brackets let it span lines):

```python
side_by_side = (
    customers_explicit.select("customer_id", F.col("customer_zip_code_prefix").alias("zip_declared"))
    .join(customers_inferred.select("customer_id", F.col("customer_zip_code_prefix").alias("zip_guessed")),
          "customer_id")
)
side_by_side.filter(F.col("zip_declared").startswith("0")).show(5)
side_by_side.filter(F.col("zip_declared").startswith("0")).count()
```

In [ ]:
# ⌨️ type it here

📝 **Observe.** Compare `zip_declared` and `zip_guessed`. What happened to the leading zero? How many customers does this affect? Nothing crashed: why is that **more** dangerous than an error?

> **I saw:** *(double-click this cell, replace this line with your own words, press Shift+Enter)*

✍️ **Now you.** 
Do the same for sellers. First look at the columns with `spark.read.csv(SELLERS, header=True).columns`. Then write a DDL string called **`SELLER_DDL`** (the zip code is an *identifier*: choose its type with Ticket 2 in mind), read the file with it into **`sellers`**, and do the three-line habit.

In [ ]:
# ✍️ your turn: write it here

📝 **Observe.** Did it work first time? Which type did you give the zip code and why? If you hit an error, what was it and how did you fix it?

> **I saw:** *(double-click this cell, replace this line with your own words, press Shift+Enter)*

In [ ]:
check("inferSchema launched jobs, an explicit schema launched none",
      lambda: j_infer.jobs > 0 and j_explicit.jobs == 0)
check("23,995 customers have a zip code starting with 0",
      lambda: side_by_side.filter(F.col("zip_declared").startswith("0")).count() == 23_995)
check("sellers: zip code is a string", lambda: dict(sellers.dtypes)["seller_zip_code_prefix"] == "string",
      "an integer drops the leading zero; a string keeps every character")
check("sellers: every zip code still has 5 characters",
      lambda: sellers.filter(F.length("seller_zip_code_prefix") != 5).count() == 0)
check("sellers: 3,095 rows", lambda: sellers.count() == 3_095)

---
## 🎫 Ticket 3 · "How many reviews do we have?"

> **Lead:** *"Quick one. Load the reviews and give me the count."*

**Issue:** customers type free text into review comments, including **line breaks** and **quote marks**. A reader that works line by line sees one review as several rows.
**See:** the count from a plain read, the rows that look wrong, and the read that fixes them.

⌨️ **Type this.** Type this into the cell below (do not paste), then run it with Shift+Enter:

```python
REVIEW_DDL = ("review_id STRING, order_id STRING, review_score INT, review_comment_title STRING, "
              "review_comment_message STRING, review_creation_date TIMESTAMP, review_answer_timestamp TIMESTAMP")

reviews_plain = spark.read.csv(REVIEWS, header=True, schema=REVIEW_DDL)
reviews_plain.count()
```

In [ ]:
# ⌨️ type it here

📝 **Observe.** Does this number look plausible to you? Write what you would have to check to know whether it is **right**.

> **I saw:** *(double-click this cell, replace this line with your own words, press Shift+Enter)*

⌨️ **Type this.** Type this into the cell below (do not paste), then run it with Shift+Enter:

```python
(reviews_plain.filter(F.col("review_score").isNull())
              .select("review_id", "order_id", "review_score")
              .show(5, truncate=48))
```

In [ ]:
# ⌨️ type it here

📝 **Observe.** These rows have no review score, so they are not reviews. What is sitting in the `review_id` column instead? In your own words: how did one review turn into two or more rows?

> **I saw:** *(double-click this cell, replace this line with your own words, press Shift+Enter)*

⌨️ **Type this.** Type this. `multiLine` lets a quoted field span lines, `quote` wraps a field, `escape` marks a quote *inside* a field:

```python
reviews = spark.read.csv(REVIEWS, header=True, schema=REVIEW_DDL,
                         multiLine=True, quote='"', escape='"')
reviews.count()
```

In [ ]:
# ⌨️ type it here

📝 **Observe.** What is the true number of reviews? By how much was the plain read wrong, and in which direction? Why would a pipeline built on the plain read still "work"?

> **I saw:** *(double-click this cell, replace this line with your own words, press Shift+Enter)*

✍️ **Now you.** 
Using `reviews`, answer two questions from the lead and store the numbers in variables: **`one_star`** = how many reviews have a score of 1, and **`with_comment`** = how many have a comment message that is not null. Print both.

In [ ]:
# ✍️ your turn: write it here

📝 **Observe.** Did it work first time? Which two methods did you combine to answer each question (for example `filter` then `count`)? If you hit an error, what was it?

> **I saw:** *(double-click this cell, replace this line with your own words, press Shift+Enter)*

In [ ]:
check("plain read: 104,162 rows (wrong!)", lambda: reviews_plain.count() == 104_162)
check("multiLine read: 99,224 reviews", lambda: reviews.count() == 99_224,
      "multiLine, quote and escape all need to be set")
check("one_star = 11,424", lambda: one_star == 11_424)
check("with_comment = 40,977", lambda: with_comment == 40_977)

---
## 🎫 Ticket 4 · "What happens when a row does not fit the schema?"

> **Lead:** *"The reviews had junk rows. What would Spark have done with them if we had declared `review_score` as an integer and trusted it?"*

**Issue:** when a row breaks your schema, Spark must do something. You choose with `mode`:

| Mode | Spark's behaviour on a row that does not fit |
|---|---|
| `PERMISSIVE` (default) | keeps the row, puts `null` where a value does not fit |
| `DROPMALFORMED` | throws the row away |
| `FAILFAST` | stops the whole read with an error |

**See:** the three modes on the plain (broken) read from Ticket 3. A repeated read is a good moment to put it in a function, as a developer would.

⌨️ **Type this.** Type this into the cell below (do not paste), then run it with Shift+Enter:

```python
def read_reviews(mode, fixed=False):
    # fixed=True -> the correct multiLine read from Ticket 3
    options = dict(multiLine=True, quote='"', escape='"') if fixed else {}
    return spark.read.csv(REVIEWS, header=True, schema=REVIEW_DDL, mode=mode, **options)


MODES = ["PERMISSIVE", "DROPMALFORMED", "FAILFAST"]

for mode in MODES:
    print(f"{mode:<14} .count() -> {read_reviews(mode).count():,}")
```

In [ ]:
# ⌨️ type it here

📝 **Observe.** All three modes gave the same number and `FAILFAST` did not fail. Were you surprised? Before you read on, write your own guess: why did Spark not notice the bad rows?

> **I saw:** *(double-click this cell, replace this line with your own words, press Shift+Enter)*

**The reason:** `.count()` does not need any column values, so Spark **never parses the columns** and never sees the bad rows. Spark skips work it does not need (you will meet this idea again in Lab 4). To force it to really look at `review_score`, ask for something that uses it:

⌨️ **Type this.** Type this into the cell below (do not paste), then run it with Shift+Enter:

```python
def rows_and_scores(df):
    r = df.agg(F.count(F.lit(1)).alias("rows"), F.count("review_score").alias("scores")).first()
    return r.rows, r.scores


def run_modes(fixed):
    out = {}
    for mode in MODES:
        try:
            out[mode] = rows_and_scores(read_reviews(mode, fixed))
            print(f"{mode:<14} rows = {out[mode][0]:>7,}   review_score filled in = {out[mode][1]:>7,}")
        except Exception as e:
            out[mode] = None
            print(f"{mode:<14} FAILED. Root cause:\n      {root_cause(e)}")
    return out


print("--- the plain (broken) read ---")
broken = run_modes(fixed=False)

print("\n--- the correct multiLine read ---")
clean = run_modes(fixed=True)
```

In [ ]:
# ⌨️ type it here

📝 **Observe.** (a) What did each mode do with the bad rows now? (b) Read the `Caused by` line of the `FAILFAST` error: which value landed in which column? (c) Why do all three modes agree on the correct read? Where does the real fix belong: in the mode, or in the read options?

> **I saw:** *(double-click this cell, replace this line with your own words, press Shift+Enter)*

✍️ **Now you.** 
Do the strict read on **orders**. Write a DDL string called **`ORDERS_DDL`** for the 8 columns (the 5 date columns are `TIMESTAMP`, the rest `STRING`; you saw the names in Ticket 1), read the file into **`orders_typed`** with `header=True`, your schema and `mode="FAILFAST"`. Then store in **`n_delivered`** how many orders have an `order_delivered_customer_date`. Use `F.count(column)` inside `agg`, so Spark really parses that column.

In [ ]:
# ✍️ your turn: write it here

📝 **Observe.** Did `FAILFAST` raise an error on orders? What does that tell you about the orders file compared with the reviews file? How many orders have a delivery date, and what do you think the others are?

> **I saw:** *(double-click this cell, replace this line with your own words, press Shift+Enter)*

In [ ]:
check("plain read + FAILFAST: raises an error", lambda: broken["FAILFAST"] is None)
check("correct read: all three modes agree on 99,224 reviews",
      lambda: all(v == (99_224, 99_224) for v in clean.values()), "go back to the multiLine read in Ticket 3")
check("orders_typed: 5 timestamp columns",
      lambda: sum(t == "timestamp" for _, t in orders_typed.dtypes) == 5, "which columns hold dates?")
check("n_delivered = 96,476", lambda: n_delivered == 96_476)

---
## Wrap up

In [ ]:
checks_summary(expected=16)

### The moves you typed in this lab
Say each one out loud, then try to type it without looking.

| Move | What it looks like |
|---|---|
| Read a CSV with its header | `spark.read.csv(path, header=True)` |
| Declare types yourself | `spark.read.csv(path, header=True, schema=DDL)` |
| Read multi-line text fields | `..., multiLine=True, quote='"', escape='"'` |
| Choose what happens to bad rows | `..., mode="FAILFAST"` |
| The three-line habit | `df.printSchema()` · `df.show(5)` · `df.count()` |
| Filter rows | `df.filter(F.col("review_score") == 1)` |
| Count non-null values | `df.agg(F.count("column")).first()[0]` |
| Pick and rename columns | `df.select("a", F.col("b").alias("c"))` |
| Join two DataFrames | `left.join(right, "key")` |
| Chain steps over several lines | `(df.filter(...).select(...).show())` |

### Before you move on
1. **Restart the kernel and Run All.** Every cell should finish without an error.
2. **Fill the Lab 1 section of `OBSERVATION_REPORT.md`:** the row-count ledger, the errors you hit, and the moves you can type from memory.
3. Commit your work:

```bash
git add notebooks/01_read.ipynb OBSERVATION_REPORT.md
git commit -m "Lab 1: read Olist correctly (header, schema, multiLine, modes)"
```

**You can say out loud, without looking:** why `inferSchema` costs a pass over the data and an explicit schema does not, why a zip code is a string, and why a row count that looks plausible can still be wrong.

*Next: Lab 2 · Transform.*